# Pure Python for Data Science & Machine Learning

Author: William

# Classification Metrics: Precision, Recall, F1 Score, ROC & AUC

Accuracy tells us the fraction of predictions we got right, but it can be **dangerously misleading** when class distributions are imbalanced. This notebook builds every major binary-classification metric from scratch in pure Python:

1. **Precision** — How many of the *predicted* positives are truly positive?
2. **Recall (Sensitivity / TPR)** — How many of the *actual* positives did we catch?
3. **F1 Score** — Harmonic mean that balances precision and recall
4. **Specificity (TNR)** — How many of the *actual* negatives did we correctly identify?
5. **ROC Curve** — Trade-off between TPR and FPR across decision thresholds
6. **AUC** — A single number summarizing ROC performance

No external libraries (no sklearn, no numpy). Just Python.

---

## 0. Why Accuracy Alone Is Not Enough

Imagine a dataset where **95 %** of patients are healthy and only **5 %** have a disease. A model that *always predicts healthy* achieves 95 % accuracy — yet it **misses every sick patient**.

| Model | Accuracy | Sick patients caught |
|-------|----------|---------------------|
| Always predict healthy | 95 % | 0 / 50 |
| Reasonable classifier  | 91 % | 48 / 50 |

The second model has *lower* accuracy but is clearly more useful. We need metrics that distinguish **types of errors**.

---

## 1. Confusion Matrix Components

For binary classification (positive = 1, negative = 0) every prediction falls into one of four bins:

|  | Predicted **Positive** | Predicted **Negative** |
|--|------------------------|------------------------|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

All the metrics in this notebook are derived from these four counts.

### Helper: compute TP, FP, TN, FN

In [ ]:
def confusion_counts(actual, predicted, positive_label=1):
    """Return (TP, FP, TN, FN) for a binary classification problem."""
    tp = fp = tn = fn = 0
    for a, p in zip(actual, predicted):
        if p == positive_label and a == positive_label:
            tp += 1
        elif p == positive_label and a != positive_label:
            fp += 1
        elif p != positive_label and a != positive_label:
            tn += 1
        else:
            fn += 1
    return tp, fp, tn, fn

In [ ]:
# Quick sanity check
actual    = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
predicted = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]

tp, fp, tn, fn = confusion_counts(actual, predicted)
print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}")
print(f"Total = {tp + fp + tn + fn} (should equal {len(actual)})")

---

## 2. Precision

**Precision** answers: *Of everything the model labelled positive, what fraction was truly positive?*

$$\text{Precision} = \frac{TP}{TP + FP}$$

**When to focus on precision:**  
When **false positives are costly**. Examples:
- **Spam detection** — Marking a legitimate email as spam is very annoying.
- **Content moderation** — Wrongly removing a post damages user trust.

High precision means: "When the model says *positive*, it is usually right."

### Python implementation

In [ ]:
def precision(actual, predicted, positive_label=1):
    """Compute precision: TP / (TP + FP)."""
    tp, fp, tn, fn = confusion_counts(actual, predicted, positive_label)
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)

In [ ]:
actual    = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
predicted = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]

p = precision(actual, predicted)
print(f"Precision: {p:.4f}")
print("Interpretation: {:.0f}% of the samples predicted positive are truly positive.".format(p * 100))

---

## 3. Recall (Sensitivity / True Positive Rate)

**Recall** answers: *Of all the actual positives, what fraction did the model catch?*

$$\text{Recall} = \frac{TP}{TP + FN}$$

**When to focus on recall:**  
When **false negatives are costly**. Examples:
- **Disease screening** — Missing a sick patient can be fatal.
- **Fraud detection** — Letting a fraudulent transaction through means real losses.

High recall means: "The model catches most of the actual positives."

### Python implementation

In [ ]:
def recall(actual, predicted, positive_label=1):
    """Compute recall (sensitivity / TPR): TP / (TP + FN)."""
    tp, fp, tn, fn = confusion_counts(actual, predicted, positive_label)
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)

In [ ]:
actual    = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
predicted = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]

r = recall(actual, predicted)
print(f"Recall: {r:.4f}")
print("Interpretation: The model catches {:.0f}% of the actual positives.".format(r * 100))

---

## 4. The Precision–Recall Trade-Off

Precision and recall are in **tension**:

- A model that predicts *everything* as positive gets **recall = 1** but **precision ≈ prevalence**.
- A model that predicts positive *only* when extremely confident gets **high precision** but **low recall**.

Lowering the decision threshold catches more positives (↑ recall) but also admits more false positives (↓ precision). We need a way to balance them.

---

## 5. F1 Score

The **F1 score** is the *harmonic mean* of precision and recall. It is high only when **both** precision and recall are high.

$$F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Why the harmonic mean instead of the arithmetic mean?**  
The harmonic mean penalizes extreme imbalances. If precision = 1.0 and recall = 0.01:
- Arithmetic mean = 0.505 (looks fine!)
- Harmonic mean = 0.0198 (reveals the problem)

**When to use F1:**  
When you need a single metric that balances precision and recall — especially with imbalanced classes where accuracy is unreliable.

### Python implementation

In [ ]:
def f1_score(actual, predicted, positive_label=1):
    """Compute the F1 score: harmonic mean of precision and recall."""
    p = precision(actual, predicted, positive_label)
    r = recall(actual, predicted, positive_label)
    if p + r == 0:
        return 0.0
    return 2 * (p * r) / (p + r)

In [ ]:
actual    = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
predicted = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]

f1 = f1_score(actual, predicted)
print(f"Precision: {precision(actual, predicted):.4f}")
print(f"Recall:    {recall(actual, predicted):.4f}")
print(f"F1 Score:  {f1:.4f}")

### Demonstrating the harmonic-mean penalty

In [ ]:
def harmonic_mean(a, b):
    if a + b == 0:
        return 0.0
    return 2 * a * b / (a + b)

print("Precision=1.0, Recall=0.01")
print(f"  Arithmetic mean: {(1.0 + 0.01) / 2:.4f}")
print(f"  Harmonic mean:   {harmonic_mean(1.0, 0.01):.4f}")
print()
print("Precision=0.8, Recall=0.8")
print(f"  Arithmetic mean: {(0.8 + 0.8) / 2:.4f}")
print(f"  Harmonic mean:   {harmonic_mean(0.8, 0.8):.4f}")

---

## 6. Specificity (True Negative Rate)

**Specificity** answers: *Of all the actual negatives, what fraction did the model correctly identify as negative?*

$$\text{Specificity} = \frac{TN}{TN + FP}$$

Specificity is the "recall for the negative class". It is the complement of the **False Positive Rate (FPR)**:

$$FPR = 1 - \text{Specificity} = \frac{FP}{FP + TN}$$

**When specificity matters:**  
- **Medical testing** — A highly specific test rarely gives false alarms.
- **Security screening** — High specificity means fewer innocent people are flagged.

### Python implementation

In [ ]:
def specificity(actual, predicted, positive_label=1):
    """Compute specificity (true negative rate): TN / (TN + FP)."""
    tp, fp, tn, fn = confusion_counts(actual, predicted, positive_label)
    if tn + fp == 0:
        return 0.0
    return tn / (tn + fp)

In [ ]:
actual    = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
predicted = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]

spec = specificity(actual, predicted)
print(f"Specificity: {spec:.4f}")
print(f"FPR:         {1 - spec:.4f}")

---

## 7. ROC Curve

The **Receiver Operating Characteristic (ROC)** curve plots **TPR** (recall) versus **FPR** (1 − specificity) as the classification threshold varies from 0 to 1.

- **X-axis:** False Positive Rate = $\frac{FP}{FP + TN}$
- **Y-axis:** True Positive Rate = $\frac{TP}{TP + FN}$

**How it works:**  
Most classifiers output a continuous *score* (probability). We convert scores to binary predictions by picking a **threshold**: predict positive if score ≥ threshold. By sweeping the threshold from 1.0 down to 0.0 we trace the ROC curve.

| Threshold | Effect |
|-----------|--------|
| Very high (→ 1.0) | Almost nothing predicted positive → low TPR, low FPR (bottom-left) |
| Very low (→ 0.0)  | Almost everything predicted positive → high TPR, high FPR (top-right) |

A **perfect** classifier has a point at (FPR=0, TPR=1) — the top-left corner.  
A **random** classifier traces the diagonal from (0,0) to (1,1).

### Pure Python: compute ROC curve data points

In [ ]:
def roc_curve(actual, scores, positive_label=1, n_thresholds=100):
    """Compute ROC curve as lists of (FPR, TPR) at evenly spaced thresholds.

    Args:
        actual:         List of true binary labels.
        scores:         List of continuous prediction scores (higher → more likely positive).
        positive_label: The label treated as the positive class.
        n_thresholds:   Number of threshold steps between 0 and 1.

    Returns:
        (fpr_list, tpr_list, thresholds) — three parallel lists.
    """
    thresholds = [i / n_thresholds for i in range(n_thresholds + 1)]
    fpr_list = []
    tpr_list = []

    for threshold in thresholds:
        predicted = [positive_label if s >= threshold else 0 for s in scores]
        tp, fp, tn, fn = confusion_counts(actual, predicted, positive_label)

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        fpr_list.append(fpr)
        tpr_list.append(tpr)

    return fpr_list, tpr_list, thresholds

### Example: ROC curve on a small dataset

In [ ]:
actual = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
scores = [0.95, 0.85, 0.70, 0.55, 0.40, 0.35, 0.30, 0.20, 0.10, 0.05]

fpr_list, tpr_list, thresholds = roc_curve(actual, scores, n_thresholds=20)

print(f"{'Threshold':>10}  {'FPR':>6}  {'TPR':>6}")
print("-" * 28)
for t, fpr, tpr in zip(thresholds, fpr_list, tpr_list):
    print(f"{t:>10.2f}  {fpr:>6.2f}  {tpr:>6.2f}")

The data above traces the ROC curve. As the threshold decreases:
- TPR rises (we catch more positives)
- FPR also rises (we also incorrectly flag more negatives)

A good model reaches high TPR while FPR is still low (the curve hugs the top-left corner).

---

## 8. AUC (Area Under the ROC Curve)

The **AUC** collapses the entire ROC curve into a single number between 0 and 1.

| AUC Value | Interpretation |
|-----------|---------------|
| 1.0 | Perfect classifier |
| 0.5 | Random guessing (diagonal ROC) |
| < 0.5 | Worse than random (labels may be flipped) |

**Probabilistic interpretation:** AUC is the probability that the model ranks a randomly chosen positive example higher than a randomly chosen negative example.

We compute it with the **trapezoidal rule** — the standard numerical integration technique:

$$AUC \approx \sum_{i=1}^{n} \frac{(x_i - x_{i-1}) \cdot (y_i + y_{i-1})}{2}$$

where $(x_i, y_i)$ are the (FPR, TPR) points sorted by FPR.

### Pure Python: trapezoidal AUC

In [ ]:
def auc_trapezoidal(fpr_list, tpr_list):
    """Compute AUC using the trapezoidal rule.

    The (FPR, TPR) points are first sorted by FPR so the integral
    proceeds left-to-right along the x-axis.
    """
    # Pair up, sort by FPR (then by TPR to break ties)
    points = sorted(zip(fpr_list, tpr_list), key=lambda pt: (pt[0], pt[1]))

    area = 0.0
    for i in range(1, len(points)):
        x_prev, y_prev = points[i - 1]
        x_curr, y_curr = points[i]
        area += (x_curr - x_prev) * (y_curr + y_prev) / 2.0

    return area

In [ ]:
area = auc_trapezoidal(fpr_list, tpr_list)
print(f"AUC: {area:.4f}")
print()
if area > 0.9:
    print("Excellent discriminative ability.")
elif area > 0.7:
    print("Good discriminative ability.")
elif area > 0.5:
    print("Fair discriminative ability.")
else:
    print("Poor — no better than random guessing.")

---

## 9. Worked Example: Applying All Metrics

Let's simulate a binary classification scenario — predicting whether a patient has a disease — and compute every metric end-to-end.

**Setup:**
- 20 patients, 8 actually have the disease (label = 1), 12 are healthy (label = 0)
- Our model outputs a probability score for each patient

In [ ]:
actual_labels = [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
model_scores  = [0.92, 0.87, 0.78, 0.72, 0.61, 0.53, 0.44, 0.30,
                 0.68, 0.45, 0.38, 0.33, 0.28, 0.22, 0.18, 0.12, 0.08, 0.05, 0.03, 0.01]

threshold = 0.5
predicted_labels = [1 if s >= threshold else 0 for s in model_scores]

print(f"Threshold: {threshold}")
print(f"Actual:    {actual_labels}")
print(f"Predicted: {predicted_labels}")

In [ ]:
tp, fp, tn, fn = confusion_counts(actual_labels, predicted_labels)

print("=== Confusion Matrix ===")
print(f"                Predicted +   Predicted -")
print(f"  Actual +        TP={tp:<4}       FN={fn}")
print(f"  Actual -        FP={fp:<4}       TN={tn}")
print()

acc = (tp + tn) / (tp + fp + tn + fn)
p   = precision(actual_labels, predicted_labels)
r   = recall(actual_labels, predicted_labels)
f1  = f1_score(actual_labels, predicted_labels)
sp  = specificity(actual_labels, predicted_labels)

print("=== Metrics ===")
print(f"  Accuracy:    {acc:.4f}")
print(f"  Precision:   {p:.4f}")
print(f"  Recall:      {r:.4f}")
print(f"  F1 Score:    {f1:.4f}")
print(f"  Specificity: {sp:.4f}")
print(f"  FPR:         {1 - sp:.4f}")

### ROC & AUC for the worked example

In [ ]:
fpr_ex, tpr_ex, thresh_ex = roc_curve(actual_labels, model_scores, n_thresholds=50)
auc_value = auc_trapezoidal(fpr_ex, tpr_ex)

print(f"AUC = {auc_value:.4f}")
print()
print("Selected ROC points:")
print(f"{'Threshold':>10}  {'FPR':>6}  {'TPR':>6}")
print("-" * 28)
for i in range(0, len(thresh_ex), 5):
    print(f"{thresh_ex[i]:>10.2f}  {fpr_ex[i]:>6.2f}  {tpr_ex[i]:>6.2f}")

### Effect of changing the threshold

In [ ]:
print(f"{'Threshold':>10}  {'Prec':>6}  {'Recall':>7}  {'F1':>6}  {'Spec':>6}  {'Acc':>6}")
print("-" * 55)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds = [1 if s >= t else 0 for s in model_scores]
    p_val  = precision(actual_labels, preds)
    r_val  = recall(actual_labels, preds)
    f1_val = f1_score(actual_labels, preds)
    sp_val = specificity(actual_labels, preds)
    tp_t, fp_t, tn_t, fn_t = confusion_counts(actual_labels, preds)
    acc_val = (tp_t + tn_t) / (tp_t + fp_t + tn_t + fn_t)
    print(f"{t:>10.1f}  {p_val:>6.3f}  {r_val:>7.3f}  {f1_val:>6.3f}  {sp_val:>6.3f}  {acc_val:>6.3f}")

Notice how:
- **Lowering** the threshold increases recall (catches more positives) but decreases precision (more false alarms).
- **Raising** the threshold increases precision but decreases recall.
- F1 peaks at the threshold that best balances the two.
- AUC stays the same regardless of threshold — it measures overall ranking quality.

---

## 10. Edge Cases and Robustness

In [ ]:
print("=== Edge case: model predicts ALL negative ===")
all_neg = [0] * 10
actual_mix = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
print(f"  Precision: {precision(actual_mix, all_neg):.4f}  (no positives predicted, defined as 0)")
print(f"  Recall:    {recall(actual_mix, all_neg):.4f}  (missed all positives)")
print(f"  F1 Score:  {f1_score(actual_mix, all_neg):.4f}")
print()

print("=== Edge case: model predicts ALL positive ===")
all_pos = [1] * 10
print(f"  Precision: {precision(actual_mix, all_pos):.4f}  (only 3/10 are truly positive)")
print(f"  Recall:    {recall(actual_mix, all_pos):.4f}  (all positives caught)")
print(f"  F1 Score:  {f1_score(actual_mix, all_pos):.4f}")
print()

print("=== Edge case: perfect classifier ===")
print(f"  Precision: {precision(actual_mix, actual_mix):.4f}")
print(f"  Recall:    {recall(actual_mix, actual_mix):.4f}")
print(f"  F1 Score:  {f1_score(actual_mix, actual_mix):.4f}")

---

## 11. ASCII ROC Visualization

Since we are not using matplotlib, here is a simple text-based ROC plot to visualize the curve shape.

In [ ]:
def ascii_roc_plot(fpr_list, tpr_list, width=40, height=20):
    """Print a simple ASCII ROC curve."""
    grid = [['.' for _ in range(width)] for _ in range(height)]

    # Draw diagonal (random classifier baseline)
    for i in range(min(width, height)):
        r = height - 1 - int(i * (height - 1) / (max(width, height) - 1))
        c = int(i * (width - 1) / (max(width, height) - 1))
        if 0 <= r < height and 0 <= c < width:
            grid[r][c] = '/'

    # Plot ROC points
    for fpr, tpr in zip(fpr_list, tpr_list):
        col = int(fpr * (width - 1))
        row = height - 1 - int(tpr * (height - 1))
        col = max(0, min(width - 1, col))
        row = max(0, min(height - 1, row))
        grid[row][col] = '*'

    print("  TPR")
    print(" 1.0 |" + "".join(grid[0]))
    for i in range(1, height - 1):
        label = f"{1.0 - i / (height - 1):.1f}" if i % 4 == 0 else "    "
        print(f" {label} |" + "".join(grid[i]))
    print(" 0.0 |" + "".join(grid[-1]))
    print("      " + "-" * width)
    print("      0.0" + " " * (width - 8) + "1.0  FPR")

print("ROC Curve (*/asterisks) vs Random Baseline (/diagonal)")
print()
ascii_roc_plot(fpr_ex, tpr_ex)

---

## Summary

| Metric | Formula | Range | Focus | When to Use |
|--------|---------|-------|-------|-------------|
| **Accuracy** | $\frac{TP+TN}{TP+FP+TN+FN}$ | $[0, 1]$ | Overall correctness | Balanced classes |
| **Precision** | $\frac{TP}{TP+FP}$ | $[0, 1]$ | Predicted positives | False positives costly (spam) |
| **Recall** | $\frac{TP}{TP+FN}$ | $[0, 1]$ | Actual positives | False negatives costly (disease) |
| **F1 Score** | $\frac{2 \cdot P \cdot R}{P + R}$ | $[0, 1]$ | Balance P & R | Imbalanced classes |
| **Specificity** | $\frac{TN}{TN+FP}$ | $[0, 1]$ | Actual negatives | False positives costly |
| **AUC** | Area under ROC | $[0, 1]$ | Ranking quality | Comparing models, threshold-free |

**Key takeaways:**

- **Accuracy** is misleading on imbalanced data — always check precision, recall, and F1.
- **Precision vs. recall** is a trade-off controlled by the decision threshold.
- **F1** is the single best metric when you cannot afford to ignore either false positives or false negatives.
- **ROC/AUC** evaluates the model's ability to rank positives above negatives across *all* thresholds.
- A high AUC means the model is good at separating the classes, regardless of what threshold you eventually pick.